In [0]:
#Import libraries.
from pyspark.sql.functions import *
from pyspark.ml.feature import *

In [0]:
df = spark.read.parquet("/Volumes/workspace/default/uber_data/clean_data")

In [0]:
#Read the cleaned dataset again
df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/workspace/default/uber_data/uber_trips_dataset_50k.csv")

In [0]:
from pyspark.sql.functions import col

df = df.filter(col("distance_km") > 0)

print(df.count())
display(df.limit(5))

49997


trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z


In [0]:

#Create new features
from pyspark.sql.functions import hour, dayofweek, month

df = df.withColumn("pickup_hour", hour("pickup_time"))

df = df.withColumn("pickup_day", dayofweek("pickup_time"))

df = df.withColumn("pickup_month", month("pickup_time"))

In [0]:
#Create Trip Type
from pyspark.sql.functions import when

df = df.withColumn(
    "trip_type",
    when(col("distance_km") < 5, "Short")
    .when(col("distance_km") < 15, "Medium")
    .otherwise("Long")
)

display(df.select("distance_km", "trip_type").limit(10))

distance_km,trip_type
2.97,Short
8.43,Medium
5.46,Medium
6.61,Medium
10.5,Medium
9.94,Medium
12.22,Medium
10.14,Medium
1.88,Short
3.7,Short


In [0]:
#Create Fare Per KM
df = df.withColumn(
    "fare_per_km",
    col("fare_amount") / col("distance_km")
)

display(df.select("fare_amount", "distance_km", "fare_per_km").limit(10))

fare_amount,distance_km,fare_per_km
10.71,2.97,3.606060606060606
22.41,8.43,2.6583629893238436
12.91,5.46,2.3644688644688645
15.7,6.61,2.3751891074130103
19.15,10.5,1.8238095238095238
19.95,9.94,2.007042253521127
25.89,12.22,2.1186579378068737
25.55,10.14,2.519723865877712
7.97,1.88,4.23936170212766
12.21,3.7,3.3000000000000003


In [0]:
#Encode City
from pyspark.ml.feature import StringIndexer

city_indexer = StringIndexer(
    inputCol="city",
    outputCol="city_index"
)

df = city_indexer.fit(df).transform(df)

display(df.select("city", "city_index").limit(10))

city,city_index
San Francisco,1.0
Boston,0.0
San Francisco,1.0
New York,4.0
Seattle,5.0
San Francisco,1.0
San Francisco,1.0
Chicago,2.0
San Francisco,1.0
Boston,0.0


In [0]:
#Encode Status
status_indexer = StringIndexer(
    inputCol="status",
    outputCol="status_index"
)

df = status_indexer.fit(df).transform(df)

display(df.select("status", "status_index").limit(10))

status,status_index
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
Completed,0.0
No-Show,2.0


In [0]:
#Encode Payment Method
payment_indexer = StringIndexer(
    inputCol="payment_method",
    outputCol="payment_index"
)

df = payment_indexer.fit(df).transform(df)

display(df.select("payment_method", "payment_index").limit(10))

payment_method,payment_index
Wallet,2.0
UPI,0.0
Cash,3.0
Wallet,2.0
Wallet,2.0
Card,1.0
UPI,0.0
Cash,3.0
Cash,3.0
UPI,0.0


In [0]:
#Create Feature Vector
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "distance_km",
        "fare_per_km",
        "pickup_hour",
        "pickup_day",
        "pickup_month",
        "city_index",
        "status_index",
        "payment_index"
    ],
    outputCol="features"
)

df = assembler.transform(df)

display(df.select("features").limit(5))

features
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.97"",""3.606060606060606"",""0.0"",""1.0"",""1.0"",""1.0"",""0.0"",""2.0""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.43"",""2.6583629893238436"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0"",""0.0""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.46"",""2.3644688644688645"",""0.0"",""1.0"",""1.0"",""1.0"",""0.0"",""3.0""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.61"",""2.3751891074130103"",""0.0"",""1.0"",""1.0"",""4.0"",""0.0"",""2.0""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""10.5"",""1.8238095238095238"",""0.0"",""1.0"",""1.0"",""5.0"",""0.0"",""2.0""]}"


In [0]:
#Final Dataset Information
print("=" * 50)
print("Feature Engineered Dataset")
print("=" * 50)

print("Rows :", df.count())
print("Columns :", len(df.columns))

display(df.limit(5))

Feature Engineered Dataset
Rows : 49997
Columns : 23


trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_hour,pickup_day,pickup_month,trip_type,fare_per_km,city_index,status_index,payment_index,features
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z,0,1,1,Short,3.606060606060606,1.0,0.0,2.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.97"",""3.606060606060606"",""0.0"",""1.0"",""1.0"",""1.0"",""0.0"",""2.0""]}"
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z,0,1,1,Medium,2.6583629893238436,0.0,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.43"",""2.6583629893238436"",""0.0"",""1.0"",""1.0"",""0.0"",""0.0"",""0.0""]}"
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z,0,1,1,Medium,2.3644688644688645,1.0,0.0,3.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.46"",""2.3644688644688645"",""0.0"",""1.0"",""1.0"",""1.0"",""0.0"",""3.0""]}"
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z,0,1,1,Medium,2.3751891074130103,4.0,0.0,2.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.61"",""2.3751891074130103"",""0.0"",""1.0"",""1.0"",""4.0"",""0.0"",""2.0""]}"
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z,0,1,1,Medium,1.8238095238095238,5.0,0.0,2.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""10.5"",""1.8238095238095238"",""0.0"",""1.0"",""1.0"",""5.0"",""0.0"",""2.0""]}"


In [0]:
#Save Feature Engineered Dataset
df.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/uber_data/feature_engineered_delta"
)

In [0]:
#Save as Parquet
df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/feature_engineered_parquet"
)

In [0]:
feature_df = spark.read.parquet(
    "/Volumes/workspace/default/uber_data/feature_engineered_dataset"
)

feature_df.coalesce(1).write.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/uber_data/feature_engineered_csv")